# 14 — Direction 1, step 5: are the replica's gradients correct and usable?

`hvac_controller/d1/grad_check.py`, then `bench.py --parts compile --compile cudagraphs`, one scenario after the other on
one GPU. Open loop (no policy yet), training configuration (torch, no float32 emulation). After a day of structured
random actions, windows start at 00:00 and at 06:00, with run commands relaxed (in [0, 1], differentiable) or exact
(fixed 0 / 1).

| part | what | gate (D1 doc §5, step 5) |
|---|---|---|
| E1 | autograd directional derivatives vs central differences (float64), h = 36 and 288, 8 random directions + one per action group, error / ‖∇J‖; one-sided differences flag directions that cross a kink; float32 autograd vs float64 | relaxed, over the directions that cross no kink: median ≤ 1e-6 and max ≤ 1e-3; float32 cos ≥ 0.999; exact: reported |
| E2 | ‖∂J/∂a_k‖ by lag from the end of a 24 h window | reported (choice of h and γ) |
| E3 | B = 1024 rows with their own action offset θ and noise σ = 0.05: first-order vs zeroth-order (score-function) gradient of θ, h = 12 … 288 | relaxed, h ≥ 36: FO SNR > 1 and ≥ 10 × ZO SNR |
| cudagraphs | `torch.compile(backend="cudagraphs")`: gradients vs eager, forward and h = 32 forward + backward speed | reported |

**Setup:** accelerator **GPU T4**; Internet on (clones the repo); attach the competition data. Expected ≈ 30–45 min.
Output: `/kaggle/working/d1_grad.zip`.

In [ ]:
REPO_URL = "https://github.com/CryAndRRich/hvac-controller"
REPO_REF = "direction1"
SCENARIOS = ["building_train_1/atlanta_2", "building_train_2/phoenix_6", "building_train_8/fairbanks_8"]
BENCH = ["--parts", "compile", "--compile", "cudagraphs", "--steps", "576", "--windows", "32",
         "--batches", "1,64,1024,4096", "--back-batches", "64,1024,4096"]

In [ ]:
import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

import torch
assert torch.cuda.is_available(), "no GPU: set the accelerator to GPU T4"
print(torch.__version__, torch.cuda.get_device_name())

REPO = Path("/tmp/hvac-controller")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REPO_REF], check=True)
COMMIT = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print(COMMIT)

SHIPPED = next(p for p in sorted(Path("/kaggle/input").rglob("training_scenarios"))
               if (p / "building_train_1" / "atlanta_2" / "config.gin").exists())
shutil.copytree(SHIPPED, REPO / "smart_buildings" / "training_scenarios", dirs_exist_ok=True)

In [ ]:
OUT = Path("/kaggle/working/d1_grad")
OUT.mkdir(parents=True, exist_ok=True)
env = dict(os.environ, PYTHONPATH=str(REPO))
tags = [s.replace("/", "__") for s in SCENARIOS]
t0 = time.time()
for scenario, tag in zip(SCENARIOS, tags):
    cfg = str(REPO / "smart_buildings" / "training_scenarios" / scenario / "config.gin")
    for part, cmd in (("grad", ["hvac_controller.d1.grad_check", str(OUT / tag / "grad"), "--config", cfg]),
                      ("bench", ["hvac_controller.d1.bench", str(OUT / tag / "bench"), "--config", cfg] + BENCH)):
        (OUT / tag / part).mkdir(parents=True, exist_ok=True)
        with open(OUT / tag / part / "log.txt", "w") as log:
            rc = subprocess.run([sys.executable, "-m"] + cmd, cwd=REPO, env=env, stdout=log,
                                stderr=subprocess.STDOUT).returncode
        text = (OUT / tag / part / "log.txt").read_text()
        print(f"[{time.time() - t0:5.0f}s] {tag} {part} exit {rc}")
        print(text[-2500:] if "Traceback" not in text else text[-8000:])

In [ ]:
def load(path):
    return json.loads(path.read_text()) if path.exists() else {"error": "no report (see log above)"}


reports = {tag: {"grad": load(OUT / tag / "grad" / "report.json"), "bench": load(OUT / tag / "bench" / "report.json")}
           for tag in tags}
(OUT / "reports.json").write_text(json.dumps({"commit": COMMIT, "reports": reports}, indent=1))
with zipfile.ZipFile("/kaggle/working/d1_grad.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file():
            z.write(p, p.relative_to(OUT.parent))
print("wrote /kaggle/working/d1_grad.zip")

e1_ok, e3_ok = [], []
for tag, rr in reports.items():
    g, b = rr["grad"], rr["bench"]
    if "starts" not in g:
        print(tag, "grad ERROR", g.get("error"))
        continue
    print(f"\n{tag}: Z={g['n_zones']} V={g['n_vavs']} actions {g['n_cont']}+{g['n_disc']} on {g['device_name']}"
          f" ({g['seconds']:.0f}s)")
    for start, res in g["starts"].items():
        for key, e1 in res["E1"].items():
            worst = max(e1["directions"].items(), key=lambda kv: kv[1]["err"])
            f32, smooth, smooth_med = e1["float32"], e1["err_max_smooth"], e1["err_median_smooth"]
            print(f"  {start} E1 {key:>12}: err median {e1['err_median']:.1e} max {e1['err_max']:.1e} (worst {worst[0]}),"
                  f" {e1['n_kink']} of {len(e1['directions'])} cross a kink, without them median "
                  f"{'-' if smooth is None else f'{smooth_med:.1e} max {smooth:.1e}'} | float32 grad cos {f32['cos']:.6f} rel {f32['rel']:.1e}"
                  f" | |grad| {e1['grad_norm']:.2e} finite {e1['grad_finite']}")
            if key.startswith("relaxed"):
                e1_ok.append(smooth is not None and smooth_med <= 1e-6 and smooth <= 1e-3 and e1["grad_finite"]
                             and f32["cos"] >= 0.999 and f32["finite"])
        for mode, e2 in res["E2"].items():
            print(f"  {start} E2 {mode:>12}: |dJ/da| by lag (rel. to peak at lag {e2['peak_lag']}): "
                  + " ".join(f"{lag}:{v:.2g}" for lag, v in e2["lag_rel"].items()))
            print(f"  {'':5}    {'':>12}  by group (lag 0 / 144): "
                  + " ".join(f"{k} {v.get('0', 0):.1e}/{v.get('144', 0):.1e}" for k, v in e2["groups"].items()))
        for key, e3 in res["E3"].items():
            fo, zo = e3["fo"], e3["zo"]
            print(f"  {start} E3 {key:>12}: FO snr {fo['snr']:8.2f} (split cos {fo['split_cos']:.3f}) | ZO snr "
                  f"{zo['snr']:6.2f} (split cos {zo['split_cos']:.2f}) | ratio {e3['snr_ratio']:7.1f} | cos(FO, ZO)"
                  f" {e3['cos_fo_zo']:.2f} | J {e3['J_mean']:.3f} ± {e3['J_std']:.1e} | finite {e3['fo_finite']}")
            if key.startswith("relaxed") and e3["h"] >= 36:
                e3_ok.append(fo["snr"] > 1.0 and e3["snr_ratio"] >= 10.0 and e3["fo_finite"])
    if "compile_diagnostics" in b:
        d = b["compile_diagnostics"].get("cudagraphs", {})
        if "error" in d:
            print(f"  cudagraphs diagnostics: {d['error'][:200]}")
        elif d:
            print(f"  cudagraphs diagnostics: reward gap {d['reward_max_abs_diff']:.1e}, non-finite grads"
                  f" {d['grad_cont_nonfinite']}/{d['grad_disc_nonfinite']}, finite grad gap {d['grad_cont_max_abs_diff_finite']}")
        for key in ("forward_cudagraphs", "backward_cudagraphs"):
            for k, v in b.get(key, {}).items():
                msg = v["error"][:150] if "error" in v else f"{v['env_steps_per_s']:,.0f} env-steps/s" + (
                    f", peak {v['peak_mem_gb']:.2f} GB, finite {v['grad_finite']}" if "grad_finite" in v else "")
                print(f"  {key:>20} {k:>10}: {msg}")
    elif "error" in b:
        print(tag, "bench ERROR", b["error"])
print(f"\nGATE E1 (relaxed, directions off kinks: FD error median <= 1e-6, max <= 1e-3; float32 cos >= 0.999):"
      f" {sum(e1_ok)}/{len(e1_ok)} pass")
print(f"GATE E3 (relaxed, h >= 36: FO SNR > 1 and >= 10x ZO): {sum(e3_ok)}/{len(e3_ok)} pass")